# PatchCore Inference Demo

This notebook demonstrates how to load a trained PatchCore model and run inference on a single image or a batch of images from the MVTec AD dataset.

**Prerequisites**: Run `python src/run_single.py --category bottle` first to generate `results/bottle/patchcore_model.pt`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

from patchcore import PatchCore
from dataset import IMAGE_TRANSFORM, MVTecTestDataset, IMAGENET_MEAN, IMAGENET_STD

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
CATEGORY = 'bottle'
DATA_ROOT = '/data/max/kaggle/anomaly_detect/anomaly_ds'
MODEL_PATH = f'../results/{CATEGORY}/patchcore_model.pt'

print(f'Device: {DEVICE}')

## Load the trained model

In [ ]:
model = PatchCore(device=DEVICE)
model.load(MODEL_PATH)
print(f'Memory bank: {len(model.memory_bank):,} vectors of dim {model.memory_bank.shape[1]}')

## Run inference on test images

In [ ]:
test_ds = MVTecTestDataset(DATA_ROOT, CATEGORY)

# Pick a few samples: first anomalous and first normal
anomaly_indices = [i for i, s in enumerate(test_ds.samples) if s[1] == 1][:3]
normal_indices  = [i for i, s in enumerate(test_ds.samples) if s[1] == 0][:3]
selected = anomaly_indices + normal_indices

results = []
for idx in selected:
    img_tensor, label, gt_mask, defect_type = test_ds[idx]
    score, amap = model.predict(img_tensor.unsqueeze(0))
    results.append((img_tensor, label, gt_mask, amap, score, defect_type))

print(f'Processed {len(results)} images')

## Visualise results

In [ ]:
def denorm(t):
    img = t.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(len(results), 4, figsize=(14, len(results) * 3.5))
if len(results) == 1:
    axes = axes[np.newaxis, :]

for row, (img_t, label, gt_mask, amap, score, defect_type) in enumerate(results):
    img_np = denorm(img_t)
    gt_np  = gt_mask.squeeze().numpy()
    
    # Normalise anomaly map for display
    amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
    heatmap = cm.jet(amap_norm)[:, :, :3]
    overlay = 0.4 * heatmap + 0.6 * img_np
    overlay = np.clip(overlay, 0, 1)

    tag   = 'ANOMALY' if label == 1 else 'NORMAL'
    color = 'red'     if label == 1 else 'green'

    axes[row, 0].imshow(img_np);               axes[row, 0].set_title(f'Input ({defect_type})', fontsize=9)
    axes[row, 1].imshow(gt_np, cmap='gray');   axes[row, 1].set_title('GT Mask', fontsize=9)
    axes[row, 2].imshow(amap, cmap='jet');     axes[row, 2].set_title('Anomaly Map', fontsize=9)
    axes[row, 3].imshow(overlay);              axes[row, 3].set_title(f'[{tag}] score={score:.3f}',
                                                                       color=color, fontsize=9)
    for ax in axes[row]:
        ax.axis('off')

plt.suptitle(f'PatchCore — {CATEGORY}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Score histogram

In [ ]:
# Run on all test images for the score distribution
from tqdm.notebook import tqdm

all_scores, all_labels = [], []
for img_t, label, gt_mask, defect_type in tqdm(test_ds, desc='Scoring'):
    s, _ = model.predict(img_t.unsqueeze(0))
    all_scores.append(s)
    all_labels.append(int(label))

import seaborn as sns
scores_np = np.array(all_scores)
labels_np = np.array(all_labels)

fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(scores_np[labels_np == 0], label='Normal',    color='green', fill=True, alpha=0.4, ax=ax)
sns.kdeplot(scores_np[labels_np == 1], label='Anomalous', color='red',   fill=True, alpha=0.4, ax=ax)
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Density')
ax.set_title(f'Score Distribution — {CATEGORY}')
ax.legend()
plt.tight_layout()
plt.show()

from metrics import compute_image_auroc
res = compute_image_auroc(all_labels, all_scores)
print(f'Image AUROC: {res["auroc"]*100:.2f}%')